# Representation Comparison Lab

```{contents}
:local:
:depth: 2
```

This lab compares array-backed and linked-node representations by counting visits and explaining tradeoffs. The goal is not to prove that one representation is always better. The goal is to connect a workload to a representation choice.


```{index} Array Search and Indexing
```

## Array Search and Indexing


Arrays are strong at indexing. Searching by value still requires checking elements until the value is found or the array ends.


In [ ]:
int[] values = { 12, 7, 19, 4, 15 };

int visits = 0;
int target = 15;
for (int index = 0; index < values.Length; index++)
{
    visits++;
    if (values[index] == target)
    {
        Console.WriteLine($"Found {target} at index {index} after {visits} visits.");
        break;
    }
}

Console.WriteLine($"Direct index values[2] = {values[2]}");


```{index} Linked Traversal
```

## Linked Traversal


A linked representation visits nodes by following references. There is no direct index jump.


In [ ]:
Node<int> head = new Node<int>(12);
head.Next = new Node<int>(7);
head.Next.Next = new Node<int>(19);
head.Next.Next.Next = new Node<int>(4);
head.Next.Next.Next.Next = new Node<int>(15);

int visits = 0;
int target = 15;
for (Node<int>? current = head; current is not null; current = current.Next)
{
    visits++;
    if (current.Value == target)
    {
        Console.WriteLine($"Found {target} after {visits} node visits.");
        break;
    }
}

public class Node<T>
{
    public T Value { get; }
    public Node<T>? Next { get; set; }
    public Node(T value) => Value = value;
}

```{index} Middle Insertion Costs
```

## Middle Insertion Costs


In an array, inserting into the middle requires shifting later elements. In a linked list, insertion after a known node changes references locally, but finding that node may still require traversal.


In [ ]:
int[] values = new int[6] { 10, 20, 40, 50, 0, 0 };
int count = 4;
int insertIndex = 2;
int shifts = 0;

for (int i = count; i > insertIndex; i--)
{
    values[i] = values[i - 1];
    shifts++;
}

values[insertIndex] = 30;
count++;

Console.WriteLine(string.Join(", ", values[..count]));
Console.WriteLine($"shifts={shifts}");


```{index} Decision Guide
```

## Decision Guide


| Workload | Better starting representation | Why |
| --- | --- | --- |
| Frequent indexing | Dynamic array | O(1) direct access by index |
| Frequent append | Dynamic array | Amortized O(1) append |
| Frequent insertion after known nodes | Linked list | O(1) reference update |
| Frequent search by value | Neither is ideal alone | Both may scan O(n) |
| Low per-item memory overhead | Dynamic array | No per-node reference field |

Use this table as a starting point, then adjust based on real operation counts and program constraints.


```{index} Lab Tasks
```

## Lab Tasks


1. Change the target in both search examples to the first, middle, last, and missing values. Record visit counts.
2. Modify the array insertion example to insert at the front. How many shifts are required?
3. Write a linked-list `CountNodes` method and test it with empty, one-node, and three-node lists.
4. Choose a representation for a playlist app where users often skip by index.
5. Choose a representation for a simulation where events are frequently inserted after a known event node.


```{index} workload; operation count; representation comparison
```

## Separate Location Cost from Update Cost

A service desk stores `n` work orders. Two requests that sound similar can require different work: “insert at position 400” supplies an index; “insert after this node” supplies a location handle. Compare the same request and the same assumptions before claiming a speed advantage.

| Request | Dynamic array | Singly linked list with head and tail |
| --- | --- | --- |
| Read index i | O(1) | O(i + 1), O(n) worst case |
| Append | O(1) amortized, O(n) on resize | O(1) with stored tail |
| Insert at front | O(n) shifts | O(1) |
| Insert by index in middle | O(n) shifts, possibly growth | O(n) locate, O(1) rewire |
| Remove by index in middle | O(n) shifts | O(n) locate, O(1) rewire |
| Remove last | O(1), without shrinking | O(n) find predecessor |
| Search unsorted values | O(n) | O(n) |
| Visit every item once | O(n) | O(n) with cursor |

The linked wrapper in this chapter exposes indices rather than public node handles. Its middle insertion therefore includes traversal. A different API could expose validated handles, but it must specify ownership and what happens when a handle refers to a removed node.

**Reason (applied):** A developer claims `Insert(400, value)` is constant-time because it changes two links. Identify the omitted work. Would adding a tail field fix it?

```{index} operation count; workload model
```

## Count a Concrete Workload

Use eight initial orders and spare array capacity. Compare a front insertion, a last-position read, and a front removal.

| Operation | Array work | Linked work |
| --- | --- | --- |
| Insert at front | Eight existing-value shifts | No link traversal; update head |
| Read last position after insertion | One indexed read | Eight link advances from head |
| Remove front | Eight existing-value shifts | No link traversal; update head |

If readers request the last position 100 times between these updates, the array uses 100 indexed reads while the linked sequence uses 800 link advances. If the workload instead performs many front updates and few index reads, the preference may reverse. Include the whole operation mix.

Counts of different primitives are not interchangeable seconds. Copying a reference, following a link, allocating a node, and checking a bound have different costs. Counts explain how work grows; benchmarks answer a narrower question about a specific implementation and machine.

**Transfer (challenge):** Compare 100 last-index reads plus one front insertion for 1,000 orders. State your assumptions about spare capacity and any stored tail. Then change the request to “read the tail value.” Explain why changing the API changes the comparison.

```{index} boundary tests; invariants; sequence contract
```

## Test the Contract Before Comparing Speed

Both implementations should produce the same logical result for the same legal operation sequence. Test correctness first. A fast implementation that loses an order is not an alternative representation of the same ADT.

| Test | Operations | Expected result |
| --- | --- | --- |
| Empty start | Construct | Count zero; read index zero throws. |
| Single element | Add A; remove index zero | Returns A; count zero. |
| Reuse after emptying | Previous test; add B | Read index zero returns B. |
| Middle insertion | Add A, C; insert B at one | A, B, C in that order. |
| Middle removal | Previous test; remove one | Returns B; leaves A, C. |
| Duplicate values | Add A twice; remove zero | One A remains. |
| Invalid insertion | Insert at count plus one | Throws; count and contents unchanged. |
| Invalid removal | Remove at count | Throws; count and contents unchanged. |

Do not test only printed output from the happy path. Check count after each operation, and verify remaining values after removal. A useful additional check compares a generated operation sequence against C# `List<T>` as a reference implementation. It does not make timing comparisons fair; it checks observable behavior.

**Design (applied):** Write the smallest operation sequence that reveals a stale-tail bug. State the expected result before running it.

```{index} memory locality; reference type; allocation
```

## Memory and Locality Qualify the Recommendation

For `int` elements, the backing array places the integer values together. For reference-type elements, the reference slots are contiguous but their target objects need not be. Sequential array access often benefits from memory locality. A linked traversal follows separately allocated node objects, which may be scattered in memory.

Linked lists also pay for a node object and link field per item. Dynamic arrays pay for spare capacity and occasional replacement arrays. Neither representation has universally smaller memory usage: the element type, spare capacity, object layout, and workload matter.

Begin with a dynamic array such as `List<T>` for ordinary indexed application sequences. Justify a linked structure when its operations and API offer a concrete benefit, such as repeated front removal or insertion at already known positions. Do not replace every `List<T>` just because some linked updates have constant cost.

**Decision memo (challenge):** Recommend a representation for a dispatch application. Give two frequent operations, one worst-case risk, a memory tradeoff, and a measurement that could change your recommendation.

```{index} summary
```

## Summary


Arrays and linked lists both represent sequences, but they optimize different operations. A strong data-structure choice starts with the workload: what the program does most often, what must be fast, and what tradeoffs are acceptable.


```{rubric} Footnotes
```
[^1]: Later chapters revisit these same tradeoffs in stacks, queues, trees, hash tables, and graphs.
